## Setup

In [1]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command


@tool
def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"


@tool
def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"


def make_agent():
    return create_agent(
        model="gpt-4o",
        tools=[read_email_tool, send_email_tool],
        checkpointer=InMemorySaver(),
        middleware=[
            HumanInTheLoopMiddleware(
                interrupt_on={
                    "send_email_tool": {
                        "allowed_decisions": ["approve", "edit", "reject"],
                    },
                    "read_email_tool": False,
                }
            ),
        ],
    )

## Approve

In [2]:
agent = make_agent()
config = {"configurable": {"thread_id": "test-approve"}}

# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config,
)
print(result["messages"])

[HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='978c5ed1-8284-4c70-ba87-68e9796f9c8d'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 97, 'total_tokens': 125, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_e68c776e99', 'id': 'chatcmpl-EWn1wjI408TZ9NVkMbjuzSABPbzd5', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a11cbf-7bb2-7313-8fd3-e9461942243a-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': 'john@test.co

In [3]:
# Step 2: Approve
if "__interrupt__" in result:
    print("⏸ Paused: Approving...")

    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config,
    )

print(f"✅ Result: {result['messages'][-1].content}")

⏸ Paused: Approving...
✅ Result: The email has been sent to john@test.com with the subject 'Hello'.


## Reject

In [4]:
agent = make_agent()
config = {"configurable": {"thread_id": "test-reject"}}

# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config,
)
print(result["messages"])

[HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='83160629-3790-45c0-a4d7-cea25bbbbc87'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 97, 'total_tokens': 125, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_e68c776e99', 'id': 'chatcmpl-EWn23mMujkVgz0opTOeAvrLOWoMKq', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a11cbf-972d-74c0-b80f-f7ad82b78d12-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': 'john@test.co

In [5]:
# Step 2: Reject
if "__interrupt__" in result:
    print("⏸ Paused: Rejecting...")

    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "reject"}
                ]
            }
        ),
        config=config,
    )

print(f"❌ Result: {result['messages'][-1].content}")

⏸ Paused: Rejecting...
❌ Result: It seems that there was an issue sending the email. Please let me know how you would like to proceed.


## Editing

In [6]:
agent = make_agent()
config = {"configurable": {"thread_id": "test-edit"}}

# Step 1: Request (with wrong info)
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'")]},
    config=config,
)
print(result["messages"])

[HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='ec671c9d-b995-4f98-a51a-6c29ffff63b0'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 95, 'total_tokens': 120, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_e68c776e99', 'id': 'chatcmpl-EWn27IQeXodaJZJKvhcBKh4Obifdj', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a11cbf-a82c-7ec0-abb1-582f56031755-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': 'wrong@email.com', 

In [7]:
# Step 2: Edit and approve
if "__interrupt__" in result:
    print("⏸ Paused: Editing...")

    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "send_email_tool",  # Tool name
                            "args": {  # New arguments
                                "recipient": "correct@email.com",
                                "subject": "Corrected Subject",
                                "body": "This was edited by human before sending",
                            },
                        },
                    }
                ]
            }
        ),
        config=config,
    )

print(f"✏️ Result: {result['messages'][-1].content}")

⏸ Paused: Editing...
✏️ Result: It seems that the email address was corrected by a human reviewer before sending. The email was sent to `correct@email.com` with the subject 'Corrected Subject'.
